# 04 — Streaming source and data-product research

A read-only research lab over one Kafka trade source per run. It connects source quality to market behavior, then turns measured evidence into candidate Bronze, Silver, Gold, and observability products.

The analysis has six parts: **Source and coverage**, **Freshness and latency**, **Uniqueness and integrity**, **Market activity**, **Cross-venue market structure**, and **Data-product evolution**. Cross-venue differences are research observations, **not executable arbitrage**; fees, depth, transfer constraints, and order latency are outside this dataset.

## 1. Configure one target

Choose `local` for the Docker broker or `msk` for the deployed AWS cluster. `quick` favors iteration; `deep` collects a longer research window. Every Kafka read remains bounded by both records and wall time.

In [ ]:
from __future__ import annotations

import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import devlab
from devlab import frames, health
from devlab.frames import NATURAL_KEY

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

In [ ]:
TARGET = "local"
RUN_MODE = "quick"
TOPIC = "md.trades.v1"
PROFILES = {
    "quick": {"limit": 20_000, "seconds": 60.0},
    "deep": {"limit": 200_000, "seconds": 600.0},
}

if TARGET not in {"local", "msk"}:
    raise ValueError("TARGET must be 'local' or 'msk'")
if RUN_MODE not in PROFILES:
    raise ValueError("RUN_MODE must be 'quick' or 'deep'")

PROFILE = PROFILES[RUN_MODE]
target = devlab.local() if TARGET == "local" else devlab.from_terraform()
target

## 2. Preflight: broker, topic, partitions, and live arrivals

Retained records answer whether the log contains data. The live-rate sample reads from `latest`, answering whether new data is arriving now. Partition watermarks expose key-distribution skew before analysis begins.

In [ ]:
topic_inventory = frames.frame(devlab.topics(target))
selected_topic = topic_inventory[topic_inventory["name"] == TOPIC]
if selected_topic.empty:
    raise RuntimeError(f"Topic {TOPIC!r} was not found on target {TARGET!r}.")

partition_watermarks = frames.frame(devlab.partitions(target, TOPIC))
live_rate = devlab.rate(target, TOPIC, seconds=10.0)
print(
    f"target={TARGET} topic={TOPIC} retained={int(selected_topic.iloc[0]['messages']):,} "
    f"live_rate={live_rate.per_second:,.1f}/s"
)
display(topic_inventory)
display(partition_watermarks)
display(
    frames.frame(
        [{"venue": venue, "trades": count} for venue, count in live_rate.by_venue.items()]
    )
)

## 3. Capture one bounded research window

The capture reads retained records from `earliest`; every downstream result uses this same window. `raw_df` retains duplicates for quality analysis, while `clean_df` removes natural-key duplicates for economic calculations.

In [ ]:
capture_started_at = datetime.now(timezone.utc)
records = devlab.collect(
    target,
    TOPIC,
    limit=PROFILE["limit"],
    seconds=PROFILE["seconds"],
    offset_reset="earliest",
)
capture_finished_at = datetime.now(timezone.utc)
raw_df = frames.trades_frame(records)
if raw_df.empty:
    raise RuntimeError(
        "No trades were captured; verify the preflight and producer before continuing."
    )
clean_df = frames.dedupe(raw_df)

event_duration = raw_df["event_ts"].max() - raw_df["event_ts"].min()
capture_summary = pd.DataFrame(
    [
        {
            "target": TARGET,
            "mode": RUN_MODE,
            "record_limit": PROFILE["limit"],
            "time_limit_s": PROFILE["seconds"],
            "captured_rows": len(raw_df),
            "deduplicated_rows": len(clean_df),
            "event_start": raw_df["event_ts"].min(),
            "event_end": raw_df["event_ts"].max(),
            "event_duration_s": event_duration.total_seconds(),
            "capture_wall_time_s": (capture_finished_at - capture_started_at).total_seconds(),
            "venues": raw_df["venue"].nunique(),
            "instruments": raw_df["instrument_id"].nunique(),
        }
    ]
)
capture_summary

## 4. Source and coverage

Profile who supplied the window, which instruments are represented, and how much economic activity each segment contains.

## 5. Freshness and latency

Separate typical latency from its tail and remember that this is exchange-to-ingest delay, not Kafka-only latency.

## 6. Uniqueness and integrity

Measure duplicates, conflicts, valid sequence gaps, partition skew, and ordering behavior before trusting market aggregates.

## 7. Market activity

Study intensity, trade sizes, directional imbalance, returns, and realized volatility on the deduplicated event-time view.

## 8. Cross-venue market structure

Compare synchronized venue VWAPs, spreads, persistence, and lagged returns without interpreting them as executable trades.

## 9. Data-product evolution

Translate named measurements into layer-specific products, validation rules, evidence strength, and priorities.